In [40]:
#pd.set_option('display.max_columns', None)
import pandas as pd
import os 
pd.set_option('display.max_rows', None)        # show all rows
pd.set_option('display.max_columns', None)     # show all columns
pd.set_option('display.width', None)           # auto-detect width
pd.set_option('display.max_colwidth', None) 

path_name = '/Users/yerik/_apple_lib/_a_progs/_a2ms_env/_9_ML_project/data/processed/df_03_27_2026_aiff_tracks_data.pkl'

df= pd.read_pickle(path_name)

df_raw= pd.read_pickle(path_name)

print(len(df))

# check if paths exist for analized songs 

print(df['Path'].apply(lambda x: os.path.exists(x)).all())


2402
True


### FILTER  = LLM + RAG

In [41]:
# ----- FILTER EXACT MATCH -----

df = df[
    (df['bpm_consistency'].round() == 100) &
    (df['dominant_bpm'] == 126) &
    (df['genre'].str.lower().str.contains('house'))
].copy()

df[['bpm_consistency', 'dominant_bpm', 'genre']].head()

,bpm_consistency,dominant_bpm,genre
4,100.0,126,Tech House
16,100.0,126,House
21,100.0,126,House
22,100.0,126,Tech House
23,100.0,126,House


In [42]:
# -----######-----######-----######-----######-----######
# DJ PLAYER (OVERLAP + CROSSFADE + THREAD SAFE)
# -----######-----######-----######-----######-----######

import pygame
import random
import threading
import time
import sys
from tqdm import tqdm


# ----- CLEAN INPUT -----
def _get_clean_input():
    try:
        import termios
        termios.tcflush(sys.stdin, termios.TCIFLUSH)
    except:
        pass

    while True:
        cmd = input("\n👉 Enter command [s/n/q]: ").strip().lower()
        if cmd != "":
            return cmd


def _audio_1104_i6_GET_df_player(df, fade_ms=4000):

    pygame.mixer.init()

    paths = df['Path'].dropna().tolist()

    if len(paths) == 0:
        print("❌ No valid paths found")
        return

    print(f"\n🎧 Loaded {len(paths)} tracks")

    # ----- STATE -----
    current_sound = {"sound": None, "channel": None}
    lock = threading.Lock()

    # ----- LOAD SOUND -----
    def load_sound(path):
        return pygame.mixer.Sound(path)

    # ----- CROSSFADE -----
    def crossfade_to(path):

        new_sound = load_sound(path)
        new_channel = new_sound.play(fade_ms=fade_ms)

        # fade out old
        if current_sound["channel"] is not None:
            current_sound["channel"].fadeout(fade_ms)

        # update current
        current_sound["sound"] = new_sound
        current_sound["channel"] = new_channel

    # ----- INIT BAR -----
    for _ in tqdm(range(50), desc="Initializing Player"):
        time.sleep(0.01)

    print("\n🎛 Controls:")
    print("   s → start")
    print("   n → next (crossfade)")
    print("   q → quit")

    # ----- MAIN LOOP -----
    while True:

        cmd = _get_clean_input()

        # ----- START -----
        if cmd in ["s", "start"]:
            path = random.choice(paths)

            with lock:
                crossfade_to(path)

            print(f"\n▶️ NOW PLAYING:\n{path}")

        # ----- NEXT -----
        elif cmd in ["n", "next"]:
            path = random.choice(paths)

            with lock:
                crossfade_to(path)

            print(f"\n⏭️ CROSSFADE TO:\n{path}")

        # ----- QUIT -----
        elif cmd in ["q", "quit"]:
            if current_sound["channel"] is not None:
                current_sound["channel"].fadeout(1000)

            print("\n🛑 Player stopped")
            break

        else:
            print("⚠️ Use: s / n / q")

In [ ]:
# !#!#!#!#! RUNNING STATEMENTS !#!#!#!#!

_audio_1104_i6_GET_df_player(df, fade_ms=5000)


🎧 Loaded 200 tracks


Initializing Player: 100%|████████████████████████████████████████████████████| 50/50 [00:00<00:00, 81.67it/s]



🎛 Controls:
   s → start
   n → next (crossfade)
   q → quit



👉 Enter command [s/n/q]:  s



▶️ NOW PLAYING:
/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_05_ULatin1_PUMA/dylu_0V[25]-126BPM-8A_Amin--id_t1-5210v---Hous-HOUSESAL--by--DJALONEAGAIN-atmyexpense(O)-2024.aiff


...


⏭️ CROSSFADE TO:
/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_09_DReynier_BDAY/dylu_0V[25]-126BPM-1A_G#min--id_t1-91000---Jack-RAPJACK--by--ANGELOFERRERIP-maddogoriginal(O)-2025.aiff



👉 Enter command [s/n/q]:  n



⏭️ CROSSFADE TO:
/Users/yerik/Music/_1_NEW_SOURCE/_2023_this/_23_03_Love_Lang/dylu_0W[23]-126BPM-7B_Fmaj--id_t11-580i---Tech-MYFAVOUR--by--DEREKMARINJULI-roomers(R)-2016.aiff



👉 Enter command [s/n/q]:  n



⏭️ CROSSFADE TO:
/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_07_4th_of_JULY_BFBFBF/dylu_0V[25]-126BPM-5B_D#maj--id_t1-730i---Tech-HEAVY--by--FSONIKBERNY-levandojerryas(R)-2012.aiff
